In [1]:
import gdspy
import numpy as np
import pandas as pd
from heapq import heappush, heappop
from matplotlib.path import Path

In [2]:
# Define the METAL layers
LAYER_METAL1 = 10
LAYER_METAL2 = 20

n_layers = 2 #Specify the number of metal layers in the GDSII that will be mapped

#Define the VIA layers

LAYER_VIA = 19

# Add obstacle layers (these should match the layers used for obstacles in your GDSII file)
OBSTACLE_LAYER1 = 13 # Replace with the actual layer numbers representing obstacles
OBSTACLE_LAYER2 = 23 # Replace with the actual layer numbers representing obstacles
OBSTACLE_LAYER12 = 15 # Replace with the actual layer numbers representing obstacles

In [3]:
def create_via_cell(lib, via_size, layer=LAYER_VIA, circular=False):
    """
    Create a cell containing a square/circular via.

    Parameters:
        lib (gdspy.GdsLibrary): The GDSII library object.
        via_size (float): Side lenth (or diameter) of the via in layout units.
        layer (int): GDSII layer number for the via.

    Returns:
        gdspy.Cell: A new cell containing the via geometry.
    """
    spacing = int(5)
    # Create a new cell for the via
    via_cell = lib.new_cell(f'{via_size}um_VIA_CELL')
    total_size = via_size + 2 * spacing
    half_size = total_size / 2
    
    # Define the via
    via_cell = lib.new_cell(f'VIA_CELL_{via_size}um_spacing_{spacing}um')
    
    # Define square via
    if not circular:
        via = gdspy.Rectangle(
            (-half_size, -half_size),  # LL corner of the via
            (half_size, half_size),  # UR corner of the via
            layer=layer  # Layer for the via 
        )
    else:
        via = gdspy.Round(
            center=(0, 0),  # Center of the via
            radius=half_size,  # Radius of the via
            number_of_points=64,  # Number of points to approximate the circle
            layer=layer  # Layer for the via 
        )
    
    # Add the via to the cell
    via_cell.add(via)
    
    return via_cell

In [4]:
def rasterize_polygon_to_grid(polygon, grid, grid_size, layer_indices):
    """
    Rasterizes a polygon onto the grid, marking the covered cells as obstacles.

    Parameters:
        polygon (list of tuples): List of (x, y) coordinates defining the polygon.
        grid (numpy.ndarray): 3D grid representing the layout.
        grid_size (float): Size of each grid cell.
        layer_indices (list of int): Indices of layers where the polygon acts as an obstacle.
    """

    # Convert polygon vertices to grid indices
    polygon_indices = [(x / grid_size, y / grid_size) for x, y in polygon]
    
    # Create a Path object for the polygon
    poly_path = Path(polygon_indices)
    
    # Determine the bounding box of the polygon
    min_x = max(int(min(idx[0] for idx in polygon_indices)), 0)
    max_x = min(int(max(idx[0] for idx in polygon_indices)), grid.shape[0] - 1)
    min_y = max(int(min(idx[1] for idx in polygon_indices)), 0)
    max_y = min(int(max(idx[1] for idx in polygon_indices)), grid.shape[1] - 1)
    
    # Mark grid cells within the polygon for specified layers
    for x in range(min_x, max_x + 1):
        for y in range(min_y, max_y + 1):
            if poly_path.contains_point((x + 0.5, y + 0.5)):
                for layer_index in layer_indices:
                    grid[x, y, layer_index] = 1  # Mark as obstacle

In [5]:
def map_obstacles_to_grid(cell, grid, grid_size):
    """
    Maps obstacle geometries from the GDSII cell onto the grid with specific layer mappings.

    Parameters:
        cell (gdspy.Cell): GDSII cell containing the layout geometry.
        grid (numpy.ndarray): 3D grid representing the layout.
        grid_size (float): Size of each grid cell.

    Returns:
        numpy.ndarray: Updated grid with obstacles marked.
    """
    if not isinstance(cell, gdspy.Cell):
        raise TypeError("Expected 'cell' to be a gdspy.Cell object.")
    if grid.ndim != 3:
        raise ValueError("Grid must be a 3D numpy array.")

    # Get polygons by layer and datatype
    polygons = cell.get_polygons(by_spec=True)
    for (layer, datatype), polys in polygons.items():
        if layer == OBSTACLE_LAYER1:  # Obstacle for the first layer
            for poly in polys:
                rasterize_polygon_to_grid(poly, grid, grid_size, [0])
        elif layer == OBSTACLE_LAYER2:  # Obstacle for the second layer
            for poly in polys:
                rasterize_polygon_to_grid(poly, grid, grid_size, [1])
        elif layer == OBSTACLE_LAYER12:  # Obstacle for both layers
            for poly in polys:
                rasterize_polygon_to_grid(poly, grid, grid_size, [0, 1])

    return grid

In [6]:
def layer_number(layer_index):
    """
    Map layer index to GDSII layer number.
    
    Parameters:
        layer_index (int): Index of the layer (e.g., 0, 1, 2).
        
    Returns:
        int: Corresponding GDSII layer number.
        
    Raises:
        ValueError: If the layer index is not mapped.
    """
    
    layer_mapping = {i : globals().get(f"LAYER_METAL{i+1}", None) for i in range(n_layers)}
    
    if layer_index not in layer_mapping:
        raise ValueError(f"Layer index {layer_index} is not mapped. Update 'layer_mapping' as needed.")
    
    return layer_mapping[layer_index]

In [7]:
def get_layer_index(layer_num, n_layers):
    """
    Map GDSII layer number to layer index, dynamically generating layers.
    
    Parameters:
        layer_num (int): The GDSII layer number to map.
        n_layers (int): Total number of layers (default: 3).
    
    Returns:
        int: The layer index corresponding to the GDSII layer number.
    
    Raises:
        ValueError: If the layer number is not indexed.
    """
    
    layer_mapping = {i: globals().get(f"LAYER_METAL{i+1}", None) for i in range(n_layers)}

    for layer_index, gds_layer_num in layer_mapping.items():
        if gds_layer_num == layer_num:
            return layer_index

    raise ValueError(f"Layer number {layer_num} is not associated with a layer index.")

In [ ]:
def a_star_route_no_overlap(start_point, end_point, grid, grid_size, width=5, via_cost=10, required_spacing=5):
    """
    A* pathfinding algorithm that avoids overlapping with existing routes.
    
    Parameters:
        width (float): Width of the wire in microns.
        required_spacing (float): Minimum spacing between wires in microns.
    """
    # Convert points to grid indices, include layer information
    start = (int(start_point[0] / grid_size), int(start_point[1] / grid_size), start_point[2])
    goal = (int(end_point[0] / grid_size), int(end_point[1] / grid_size), end_point[2])
    
    def heuristic(a, b):
        return abs(a[0] - b[0]) + abs(a[1] - b[1]) + (via_cost if a[2] != b[2] else 0)

    neighbors = [
        (0, 1, 0), (0, -1, 0), (-1, 0, 0), (1, 0, 0),
        (-1, -1, 0), (1, 1, 0), (-1, 1, 0), (1, -1, 0),
        (0, 0, 1), (0, 0, -1)
    ]
    
    open_set = []
    heappush(open_set, (0, start))
    came_from = {}
    gscore = {start: 0}
    fscore = {start: heuristic(start, goal)}
    close_set = set()

    # Calculate buffer based on wire width and required spacing
    wire_half_width = width / 2
    buffer_radius = wire_half_width + required_spacing
    spacing_buffer = int(buffer_radius / grid_size)
    
    while open_set:
        current = heappop(open_set)[1]

        if current == goal:
            path = []z
            while current in came_from:
                path.append(current)
                current = came_from[current]
            path.append(start)
            path.reverse()
            
            # Mark the path and buffer as obstacles
            for x, y, z in path:
                grid[x, y, z] = 1
                for dx in range(-spacing_buffer, spacing_buffer + 1):
                    for dy in range(-spacing_buffer, spacing_buffer + 1):
                        if dx**2 + dy**2 <= spacing_buffer**2:
                            nx, ny = x + dx, y + dy
                            if 0 <= nx < grid.shape[0] and 0 <= ny < grid.shape[1]:
                                grid[nx, ny, z] = 1  # Mark obstacle layer

                                #visualize buffer route blocks here
                                
            return path
        
        close_set.add(current)
        
        for dx, dy, dz in neighbors:
            neighbor = (current[0] + dx, current[1] + dy, current[2] + dz)
            tentative_g_score = gscore[current] + (via_cost if dz != 0 else 
            2 if dx != 0 and dy != 0 else 1)

            if not (0 <= neighbor[0] < grid.shape[0] and
                    0 <= neighbor[1] < grid.shape[1] and
                    0 <= neighbor[2] < grid.shape[2]):
                continue
            if grid[neighbor[0], neighbor[1], neighbor[2]] == 1:
                continue
            if neighbor in close_set:
                continue

            if tentative_g_score < gscore.get(neighbor, float('inf')):
                came_from[neighbor] = current
                gscore[neighbor] = tentative_g_score
                fscore[neighbor] = tentative_g_score + heuristic(neighbor, goal)
                heappush(open_set, (fscore[neighbor], neighbor))
    return None

def route_paths(point_pairs, grid, grid_size, via_cell, lib, route_cell):
    """
    Route paths from point_pairs sequentially, ensuring no overlap.
    """
    for point_a, point_b, width in point_pairs:
        path_points = a_star_route_no_overlap(point_a, point_b, grid, grid_size, width)
        if not path_points:
            continue
        
        routing_elements = generate_path_geometry_multi_layer(path_points, width, via_cell)
        for elem in routing_elements:
            route_cell.add(elem)
    return route_cell

In [9]:
def generate_path_geometry_multi_layer(path_points, width, via_cell):
    """
    Generate gdspy path geometry from 3D path points, handling layers and vias.
    """
    
    if not path_points or len(path_points) < 2:
        return None
    print("path points length: ", len(path_points))
    elements = []
    current_layer = path_points[0][2]
    path_coords = [(path_points[0][0], path_points[0][1])]
    
    path = gdspy.FlexPath(path_coords, width, layer=layer_number(current_layer), ends="round", corners="round")#should only have one point
    for i in range(1, len(path_points)):
        point = path_points[i]
        layer = point[2]

        if layer == current_layer:
            # Continue on the same layer
            path.segment([(point[0], point[1])])
        else:
            # Different layer: finish current path, add via, start new path
            elements.append(path)
            via_position = (point[0], point[1])
            via_ref = gdspy.CellReference(via_cell, origin=via_position)
            elements.append(via_ref)
            path = gdspy.FlexPath([via_position], width, layer=layer_number(layer), ends="round", corners="round")
            current_layer = layer
    if len(path.points) > 1:
         elements.append(path)
         print("flexpath length: ", len(path.points))
    return elements

In [10]:
def route_paths(point_pairs, grid, grid_size, via_cell, lib, route_cell):
    """
    Route paths from point_pairs sequentially, ensuring no overlap.
    """
    for point_a, point_b, width in point_pairs:
        print(f"Routing with A* from {point_a} to {point_b} with width {width}...")
        path_points = a_star_route_no_overlap(point_a, point_b, grid, grid_size)
        print("running a star completed")
        if path_points is None or len(path_points) < 2:
            print(f"No path found between {point_a} and {point_b}")
            continue
               
                

        # Generate path geometry and add to the routing cell
        print("generating path")
        routing_elements = generate_path_geometry_multi_layer(path_points, width, via_cell)
        if routing_elements:
            for elem in routing_elements:
                if isinstance(elem, gdspy.CellReference) or len(elem.points) > 1:
                    route_cell.add(elem)
            # print(f"Route successfully created between {point_a} and {point_b}.")
        else:
            print(f"Failed to generate geometry for route between {point_a} and {point_b}.")
    return route_cell

In [11]:
def main():
    # Load the Excel file
    excel_file = 'PIN-LIST_3.xlsx'
    sheet_name = 'Pin-List'
    df = pd.read_excel(excel_file, sheet_name=sheet_name)

    # Extract point pairs
    point_pairs = []
    for _, row in df.iterrows():
        point_a = (row['X1'], row['Y1'], get_layer_index(row['LAYER1'], n_layers))
        point_b = (row['X2'], row['Y2'], get_layer_index(row['LAYER2'], n_layers))
        width = row['LINE-WIDTH']
        point_pairs.append((point_a, point_b, width))

    # Initialize grid and library
    grid_size = 1
    grid_width = 25000 // grid_size
    grid_height = 25000 // grid_size
    num_layers = 2
    grid = np.zeros((grid_width, grid_height, num_layers), dtype=int)
    print(grid.shape)

    lib = gdspy.GdsLibrary()
    route_cell = lib.new_cell('ROUTING')

    # Create a sample via cell
    print("Creating via cell")
    via_cell = create_via_cell(lib, via_size=1, circular=True)
     

    # Route paths ensuring no overlap
    print("running route_paths")
    route_cell = route_paths(point_pairs, grid, grid_size, via_cell, lib, route_cell)
    print(grid)
    
    for path in route_cell.paths:
        print("final path lengths: ", len(path.points))
        # if len(path.points) <2:
        #     route_cell.polygons.remove(path)
        
    # Save the output GDS file
    output_gds = 'output_routing_no_overlap_deepv1.gds'
    lib.write_gds(output_gds)
    
    print(f"Routing completed. Output saved to {output_gds}")

In [12]:
if __name__ == '__main__':
    main()

(25000, 25000, 2)
Creating via cell
running route_paths
Routing with A* from (2922.045581250871, 12637.137009782638, 0) to (3631.375906531899, 12677.572824014307, 0) with width 5...
running a star completed
generating path
path points length:  710
flexpath length:  710
Routing with A* from (2922.135757525609, 12627.13741637893, 0) to (3631.3793870608606, 12667.572824620012, 0) with width 5...
running a star completed
generating path
path points length:  710
flexpath length:  710
Routing with A* from (2922.225933800347, 12617.137822975223, 0) to (3631.382867589823, 12657.572825225714, 0) with width 5...
running a star completed
generating path
path points length:  710
flexpath length:  710
Routing with A* from (2922.316110075085, 12607.138229571516, 0) to (3631.3863481187855, 12647.57282583142, 0) with width 5...
running a star completed
generating path
path points length:  710
flexpath length:  710
Routing with A* from (2922.406286349823, 12597.138636167807, 0) to (3631.389828647747, 1